**© Copyright AIDENTIFY. All rights reserved.**

# Part 3 | Session 25: vLLM 서빙 — PagedAttention과 OpenAI 호환 API

---

### 📋 학습 목표

- 🔹 vLLM의 핵심 기술(PagedAttention, Continuous Batching)을 이해합니다
- 🔹 vLLM Python API로 오프라인 추론을 실행합니다
- 🔹 vLLM OpenAI 호환 API 서버를 로컬에서 구축합니다
- 🔹 동일 모델을 Transformers 직접 추론과 vLLM 서빙으로 처리량 비교합니다
- 🔹 GPU VRAM에 맞는 vLLM 설정(모델 / dtype / max-model-len)을 선택합니다

### 📦 필요 라이브러리

```
vllm, openai, torch, transformers, requests
```

### ⏱️ 예상 소요 시간: 약 90분

### 🖥️ 실습 환경

| 환경 | GPU | 실습 모델 | 비고 |
|------|-----|-----------|------|
| RTX 4060 (8GB) | 1.5B FP16 | Qwen2.5-1.5B-Instruct | 본 과정 기본 |
| RTX 4090 / TITAN RTX (24GB+) | 7B FP16 | Qwen2.5-7B-Instruct | 멀티GPU 가능 |

---

> 💡 Session 21~24의 양자화 모델(AWQ/GPTQ/QLoRA)을 **production에서 어떻게 서빙하는가**가 본 세션의 주제입니다.

---

## 1️⃣ vLLM 소개

**vLLM**은 대규모 언어 모델을 빠르게 서빙하기 위한 오픈소스 추론 엔진입니다 (UC Berkeley 2023).

### 🚀 핵심 기술

- 🔹 **PagedAttention** — GPU 메모리를 페이지 단위로 관리해 KV cache 메모리 낭비를 **90% 이상** 절감
- 🔹 **Continuous Batching** — 요청이 끝나는 즉시 빈 슬롯에 다음 요청 투입 → 대기시간 최소화
- 🔹 **OpenAI 호환 API** — `openai` 라이브러리로 바로 호출 (기존 코드 그대로 재사용)
- 🔹 **양자화 통합** — AWQ / GPTQ / FP8 / Marlin 모델 직접 서빙

### 📊 Static vs Continuous Batching

```
Static Batching (기존):
  요청1: [████████████████████]  → 모든 요청이 끝날 때까지 대기
  요청2: [████████____대기____]  → 짧은 응답도 긴 응답 끝까지 대기
  요청3: [██████████████______]  → GPU 활용률 낮음

Continuous Batching (vLLM):
  요청1: [████████████████████]  → 끝나면 즉시 새 요청 투입
  요청2: [████████] → 요청4: [████████████]
  요청3: [██████████████] → 요청5: [██████]   → GPU 항상 바쁨
```

---

In [ ]:
# 📊 vLLM vs Ollama vs Transformers 서빙 비교
print("📊 LLM 서빙 프레임워크 비교")
print("=" * 78)

comparisons = [
    ("설치 난이도",   "보통",            "매우 쉬움",       "매우 쉬움"),
    ("처리량 (QPS)",  "높음 (10x+)",     "보통 (3x)",       "기준 (1x)"),
    ("동시 요청",     "수백 개",          "수십 개",          "1개"),
    ("배칭",          "Continuous",      "Static",          "없음"),
    ("모델 포맷",     "HF (FP16/AWQ)",   "GGUF (양자화)",   "HF (모든 형식)"),
    ("API 호환",      "OpenAI",          "OpenAI",          "없음"),
    ("최소 VRAM",     "6GB+ (1.5B)",     "4GB+ (1.5B)",     "2GB+"),
    ("주요 용도",     "프로덕션 서빙",    "개발/테스트",      "실험/학습"),
]

print(f"{'항목':<15} {'vLLM':>17} {'Ollama':>17} {'Transformers':>17}")
print("-" * 78)
for row in comparisons:
    print(f"{row[0]:<15} {row[1]:>17} {row[2]:>17} {row[3]:>17}")

print(f"\n💡 선택 가이드:")
print(f"   🔹 프로덕션 서빙 (다수 동시 사용자) → vLLM")
print(f"   🔹 로컬 개발/테스트            → Ollama")
print(f"   🔹 연구/커스텀 추론              → Transformers")

---

## 2️⃣ vLLM 설치 & 환경 확인

vLLM은 pip으로 설치합니다. **CUDA 12.1 이상**, **Python 3.9+** 가 필요합니다.

```bash
# 터미널에서
pip install vllm
```

> ⚠️ 설치 시간이 길 수 있습니다 (5~10분).
> 설치 실패 시 CUDA / Python 버전을 확인하세요.

In [ ]:
# 🔧 vLLM 설치 확인 + GPU 환경 점검
import sys, torch

print("🔧 환경 점검")
print("=" * 50)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

vllm_available = False
default_model = None
vram_gb = 0

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f}GB")

    if vram_gb >= 20:
        default_model = "Qwen/Qwen2.5-7B-Instruct"
        print(f"\n✅ VRAM {vram_gb:.0f}GB → 7B FP16 서빙 가능")
    elif vram_gb >= 6:
        default_model = "Qwen/Qwen2.5-1.5B-Instruct"
        print(f"\n✅ VRAM {vram_gb:.0f}GB → 1.5B FP16 서빙 가능")
    else:
        print(f"\n⚠️ VRAM {vram_gb:.0f}GB → vLLM 사용 어려움")

    if default_model:
        print(f"📌 기본 실습 모델: {default_model}")
else:
    print("⚠️ GPU를 사용할 수 없습니다.")

print()
try:
    import vllm
    print(f"✅ vLLM: {vllm.__version__}")
    vllm_available = True
except ImportError:
    print("❌ vLLM이 설치되어 있지 않습니다.")
    print("   터미널: pip install vllm")

---

## 3️⃣ vLLM 사용 방식: 오프라인 추론 vs API 서버

vLLM은 **두 가지 방식**으로 사용할 수 있습니다.

### 방식 A: 오프라인 추론 (Python API)

코드 안에서 직접 모델을 로드하고 추론. 서버 불필요.

```python
from vllm import LLM, SamplingParams
llm = LLM(model="Qwen/Qwen2.5-7B-Instruct", dtype="float16")
outputs = llm.generate(["한국의 수도는?", "파이썬이란?"], SamplingParams(max_tokens=128))
```

- 코드 간단, 스크립트/노트북 실험에 적합
- 배치 처리 효율적
- **단일 프로세스만** (외부 호출 불가)

### 방식 B: API 서버 (OpenAI 호환)

터미널에서 서버 띄우고 HTTP API로 호출.

```bash
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-7B-Instruct --dtype float16 --port 9100
```

```python
from openai import OpenAI
client = OpenAI(base_url="http://localhost:9100/v1", api_key="not-needed")
response = client.chat.completions.create(model="...", messages=[...])
```

- OpenAI API와 **동일 인터페이스** → 기존 코드 재사용
- Continuous Batching으로 동시 요청 자동 처리
- 여러 클라이언트 동시 접속 가능 (production 적합)

```
⚠️ 같은 GPU에서 A·B를 동시에 실행 불가!
   → 방식 A 실습 후 커널 재시작 → 방식 B 실습
```

> 본 노트북은 **A → B 순서**로 실습합니다.

---

In [ ]:
# 🚀 vLLM 오프라인 추론 — 모델 로딩 + 단일 프롬프트
import time

if not vllm_available:
    print("⚠️ vLLM 미설치 — 셀 건너뜀")
elif not torch.cuda.is_available():
    print("⚠️ GPU 미사용 — 셀 건너뜀")
else:
    from vllm import LLM, SamplingParams

    if vram_gb >= 20:
        model_name = "Qwen/Qwen2.5-7B-Instruct"
        max_len, mem_util = 2048, 0.85
    else:
        model_name = "Qwen/Qwen2.5-1.5B-Instruct"
        max_len, mem_util = 1024, 0.80

    print(f"🚀 모델 로딩: {model_name}")
    print(f"   max_model_len={max_len}, gpu_memory_utilization={mem_util}")

    start = time.time()
    llm = LLM(
        model=model_name,
        dtype="float16",
        max_model_len=max_len,
        gpu_memory_utilization=mem_util,
        enforce_eager=True,  # Turing(TITAN RTX 등)에서 FA2 미지원 시 필요
    )
    print(f"✅ 로딩 완료 ({time.time()-start:.1f}초)")

    sampling = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=128)
    prompts = ["인공지능의 미래에 대해 간단히 설명해주세요."]

    print("\n📝 단일 프롬프트 추론")
    s = time.time()
    outputs = llm.generate(prompts, sampling)
    elapsed = time.time() - s

    for o in outputs:
        gen = o.outputs[0].text
        n = len(o.outputs[0].token_ids)
        print(f"\n❓ {o.prompt}")
        print(f"💬 {gen}")
        print(f"\n📊 {n} tokens / {elapsed:.2f}s = {n/elapsed:.1f} tok/s")

In [ ]:
# 📊 배칭 성능 비교: 개별 추론(5회) vs 배치 추론(5개 동시)
if not vllm_available or not torch.cuda.is_available():
    print("⚠️ 건너뜀")
else:
    try:
        llm
    except NameError:
        print("⚠️ 이전 셀에서 llm을 먼저 로드하세요.")
    else:
        sampling = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=128)
        batch_prompts = [
            "파이썬의 장점을 3가지 알려주세요.",
            "머신러닝과 딥러닝의 차이점은?",
            "한국의 대표적인 전통 음식 3가지는?",
            "클라우드 컴퓨팅이란 무엇인가요?",
            "좋은 코드를 작성하는 방법은?",
        ]

        print("📊 배칭 성능 비교")
        print("=" * 60)

        # 1) 개별 (1개씩 × 5)
        s = time.time()
        ind_tok = 0
        for p in batch_prompts:
            o = llm.generate([p], sampling)
            ind_tok += len(o[0].outputs[0].token_ids)
        ind_t = time.time() - s

        # 2) 배치 (5개 한 번에)
        s = time.time()
        bouts = llm.generate(batch_prompts, sampling)
        bat_t = time.time() - s
        bat_tok = sum(len(o.outputs[0].token_ids) for o in bouts)

        print(f"\n🔸 개별 추론 (1×5): {ind_t:.2f}s, {ind_tok} tok, {ind_tok/ind_t:.1f} tok/s")
        print(f"🔹 배치 추론 (5동시): {bat_t:.2f}s, {bat_tok} tok, {bat_tok/bat_t:.1f} tok/s")
        print(f"\n🚀 배치 처리 속도 향상: {ind_t/bat_t:.1f}x")
        print("\n💡 Continuous Batching이 GPU 활용률을 높여 전체 처리량을 크게 증가시킵니다.")

In [ ]:
# 🧹 GPU 메모리 정리 — 다음 섹션(API 서버)을 위해 반드시 실행
import gc

try:
    del llm
    print("✅ vLLM 모델 해제")
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"✅ GPU 메모리 정리 완료 (현재: {allocated:.1f}GB)")

print("\n⚠️ 같은 GPU에서 다음 섹션의 API 서버를 띄우려면")
print("   여기서 끝나고 커널 재시작(Kernel → Restart) 권장.")

---

## 4️⃣ vLLM OpenAI 호환 API 서버

서버를 띄우면 `openai` 라이브러리로 바로 호출 가능합니다.

### 🔄 서버 방식의 장점

- 모델을 **한 번만 로드** → 여러 클라이언트가 공유
- 동시 요청 자동 배칭 → 높은 처리량
- OpenAI 호환 → 기존 코드 그대로 사용

### ⚠️ 아래 셀 실행 전 순서

```
┌──────────────────────────────────────────────────────────────┐
│ Step 1. 노트북 커널 재시작 (Kernel → Restart Kernel)            │
│         → 오프라인 추론에서 사용한 GPU 메모리 해제              │
│                                                              │
│ Step 2. 터미널에서 vLLM API 서버 실행 (다음 셀 참고)            │
│                                                              │
│ Step 3. "Uvicorn running on ..." 메시지 확인 후                │
│         아래 코드 셀 실행                                       │
└──────────────────────────────────────────────────────────────┘
```

### 📋 Step 2 서버 실행 명령어

**VRAM 20GB+ (RTX 4090 / A100 등):**
```bash
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-7B-Instruct \
    --dtype float16 \
    --max-model-len 2048 \
    --gpu-memory-utilization 0.85 \
    --enforce-eager \
    --port 9100
```

**RTX 4060 (8GB):**
```bash
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --dtype float16 \
    --max-model-len 1024 \
    --gpu-memory-utilization 0.80 \
    --enforce-eager \
    --port 9100
```

> 서버 시작까지 **수 분** 걸립니다. `Uvicorn running on http://0.0.0.0:9100` 메시지를 기다리세요.

---

In [ ]:
# ⚙️ vLLM API 서버 주요 파라미터
params = [
    ("--model",                   "HuggingFace 모델 이름",           "필수"),
    ("--dtype",                   "데이터 타입 (float16/bfloat16/auto)", "auto"),
    ("--max-model-len",           "최대 시퀀스 길이",                  "모델 기본값"),
    ("--gpu-memory-utilization",  "GPU 메모리 사용 비율 (0~1)",       "0.9"),
    ("--tensor-parallel-size",    "Tensor 병렬 GPU 수 (멀티GPU)",     "1"),
    ("--max-num-seqs",            "동시 처리 최대 시퀀스 수",          "256"),
    ("--quantization",            "양자화 (awq / gptq / squeezellm)", "없음"),
    ("--port",                    "API 서버 포트",                    "8000"),
    ("--host",                    "바인딩 호스트",                    "0.0.0.0"),
    ("--served-model-name",       "API에서 사용할 모델 별칭",          "--model과 동일"),
    ("--enforce-eager",           "FlashAttention 비활성화 (구형 GPU)", "False"),
]

print("⚙️ vLLM API 서버 주요 파라미터")
print("=" * 78)
print(f"{'파라미터':<28} {'설명':<32} {'기본값':>15}")
print("-" * 78)
for p, d, v in params:
    print(f"{p:<28} {d:<32} {v:>15}")

In [ ]:
# 📡 OpenAI 클라이언트로 vLLM 서버 호출
# ⚠️ 실행 전: 터미널에서 vLLM 서버를 먼저 띄우세요 (위 명령어 참고)

import requests, time

VLLM_BASE_URL = "http://localhost:9100/v1"
vllm_server_running = False
print("🔍 vLLM 서버 연결 확인 중...")

for attempt in range(5):
    try:
        resp = requests.get("http://localhost:9100/health", timeout=5)
        if resp.status_code == 200:
            print("✅ vLLM 서버가 실행 중입니다.")
            vllm_server_running = True
            break
    except requests.ConnectionError:
        pass
    if attempt < 4:
        print(f"   재시도 {attempt+1}/5...")
        time.sleep(5)

if not vllm_server_running:
    print("\n❌ vLLM 서버에 연결할 수 없습니다.")
    print("   터미널에서 위 명령어로 서버를 먼저 띄우세요.")
else:
    from openai import OpenAI
    client = OpenAI(base_url=VLLM_BASE_URL, api_key="not-needed")

    models = client.models.list()
    model_id = models.data[0].id
    print(f"📌 서빙 중인 모델: {model_id}")

    start = time.time()
    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": "당신은 유능한 AI 어시스턴트입니다."},
            {"role": "user",   "content": "파이썬의 장점을 3가지 알려주세요."},
        ],
        temperature=0.7,
        max_tokens=256,
    )
    elapsed = time.time() - start

    print(f"\n💬 응답:\n{response.choices[0].message.content}")
    print(f"\n📊 응답 시간: {elapsed:.2f}s")
    print(f"   토큰: 입력 {response.usage.prompt_tokens}, 출력 {response.usage.completion_tokens}")

In [ ]:
# 🌊 스트리밍 응답 — vLLM도 OpenAI와 동일하게 stream=True
if not vllm_server_running:
    print("⚠️ 서버가 실행 중이 아닙니다.")
else:
    print("🌊 스트리밍 응답 테스트")
    print("=" * 50)
    print("❓ 질문: 딥러닝이 무엇인지 초보자에게 설명해주세요.\n")
    print("💬 응답: ", end="", flush=True)

    start = time.time()
    token_count = 0
    first_token_time = None

    stream = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": "딥러닝이 무엇인지 초보자에게 설명해주세요."}],
        temperature=0.7,
        max_tokens=200,
        stream=True,
    )

    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            if first_token_time is None:
                first_token_time = time.time() - start
            print(delta, end="", flush=True)
            token_count += 1

    total = time.time() - start
    print(f"\n\n📊 성능:")
    print(f"   첫 토큰 (TTFT): {first_token_time:.3f}s")
    print(f"   총 시간:        {total:.2f}s")
    print(f"   토큰 수:        {token_count}개")
    print(f"   처리 속도:      {token_count/total:.1f} tok/s")

---

## 5️⃣ GPU별 vLLM 추천 설정

GPU VRAM에 따라 서빙 가능한 모델과 권장 설정이 달라집니다.

---

In [ ]:
# 📋 GPU별 vLLM 모델/설정 가이드
print("📋 GPU별 vLLM 추천 설정")
print("=" * 78)

gpu_configs = [
    {
        "gpu": "RTX 4060 (8GB)",
        "models": [("Qwen2.5-1.5B-Instruct", "float16", 1024, 0.80)],
        "note": "1.5B까지 가능, 3B 이상은 VRAM 부족",
    },
    {
        "gpu": "RTX 4090 (24GB)",
        "models": [
            ("Qwen2.5-7B-Instruct",     "float16", 4096, 0.85),
            ("Qwen2.5-7B-Instruct-AWQ", "auto",    8192, 0.85),
        ],
        "note": "7B FP16 또는 7B AWQ 양자화",
    },
    {
        "gpu": "TITAN RTX (24GB)",
        "models": [("Qwen2.5-7B-Instruct", "float16", 2048, 0.85)],
        "note": "Turing 아키텍처 — enforce-eager 필요 (FA2 미지원)",
    },
    {
        "gpu": "A100 (40GB / 80GB)",
        "models": [
            ("Qwen2.5-14B-Instruct",     "float16", 8192, 0.90),
            ("Qwen2.5-72B-Instruct-AWQ", "auto",    4096, 0.90),
        ],
        "note": "프로덕션 권장 GPU",
    },
]

for c in gpu_configs:
    print(f"\n🔹 {c['gpu']}")
    print(f"   {c['note']}")
    for model, dtype, max_len, mem in c["models"]:
        print(f"   → {model} (dtype={dtype}, max_len={max_len}, mem={mem})")

# 현재 GPU에 맞는 추천
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)
    print(f"\n" + "=" * 78)
    print(f"📌 현재 GPU: {gpu_name} ({vram_gb:.0f}GB)")
    if vram_gb >= 20:
        print(f"   → 7B FP16 서빙 추천")
    elif vram_gb >= 6:
        print(f"   → 1.5B FP16 서빙 추천")
    else:
        print(f"   → vLLM 대신 Ollama 사용 추천")

---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

| 주제 | 핵심 내용 |
|------|----------|
| PagedAttention | KV cache 메모리 낭비 90%+ 절감 |
| Continuous Batching | 빈 슬롯 즉시 재활용 → GPU 항상 바쁨 |
| 오프라인 추론 | `LLM().generate(prompts, SamplingParams)` |
| API 서버 | `python -m vllm.entrypoints.openai.api_server` |
| OpenAI 호환 | 기존 `openai` 코드 그대로 — `base_url`만 교체 |
| GPU별 설정 | RTX 4060 = 1.5B, RTX 4090/TITAN = 7B, A100 = 14~72B |

### 양자화 + vLLM 페어링 (Session 22~24 연결)

```
Session 22~24:  HF 모델  ──AWQ/GPTQ──▶  4bit 모델
                                            │
Session 25 (지금):                          ▼
                          vLLM (--quantization awq)
                                            │
                                            ▼
                      🚀 production 서빙 (OpenAI 호환 API)
```

### 다음 세션 예고

- 🔜 **Session 26**: RAG 기본개념과 파이프라인 — vLLM으로 띄운 LLM 백엔드 위에서 검색 증강 생성을 구현합니다.

---

### 💡 실습 과제

1. vLLM Python API로 Qwen2.5-1.5B-Instruct 오프라인 추론 실행
2. vLLM API 서버 띄우고 OpenAI 클라이언트로 호출
3. Ollama와 vLLM에 같은 프롬프트 5개씩 보내 응답 시간 비교
4. (심화) `--quantization awq` 옵션으로 AWQ 양자화 모델 서빙 (Session 22~24 결과 활용)

### 📚 참고 자료
- [vLLM 공식 문서](https://docs.vllm.ai/)
- [vLLM GitHub](https://github.com/vllm-project/vllm)
- [PagedAttention 논문](https://arxiv.org/abs/2309.06180)

---

In [ ]:
print("✅ Session 25 완료!")
print("📚 다음 세션: RAG 기본개념과 파이프라인 (Part 4 시작)")